[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch13-model-serving/01-hang-doi-va-du-dia.ipynb)

# Hàng đợi, độ trễ đuôi và dư địa

Chương 13 nói một dịch vụ trực tuyến được chấm bằng việc **đủ tỉ lệ yêu cầu hoàn thành trước hạn
chót**, và hàng đợi là thứ làm điều đó khó. Một hệ thống còn trong ngân sách độ trễ ở lưu lượng vừa
phải có thể vượt ngân sách sau một đợt tăng nhỏ, vì độ trễ không tăng tuyến tính theo mức sử dụng
(utilization). Chương đưa hai công cụ: định luật Little và hàng đợi M/M/1.

Sổ tay này dựng một hàng đợi thật, bằng mô phỏng: yêu cầu đến theo Poisson, một máy phục vụ từng
yêu cầu theo thứ tự đến. Rồi nó đặt kết quả cạnh các công thức của chương, và đi xa hơn chương ở
những chỗ chương chỉ nói bằng lời: nhiều máy, toả ra nhiều máy (fan-out), và một đợt tăng vọt.

**Bạn sẽ làm:**
1. tính định luật Little của napkin math 1.2, rồi kiểm nó trên một hàng đợi mô phỏng;
2. mô phỏng hàng đợi M/M/1 và M/D/1, đặt độ trễ trung bình và p99 cạnh phương trình 5 và 6;
3. tính lại hai ngộ nhận và một cạm bẫy ở cuối chương;
4. làm lại napkin math 1.3 (hoạch định năng lực cho ResNet-50), rồi kiểm nó bằng mô phỏng và xem
   chính sách phân phối yêu cầu thay đổi kết quả ra sao;
5. tính xác suất một yêu cầu toả ra nhiều máy gặp ít nhất một phản hồi chậm;
6. cho một đợt tăng vọt đi qua hệ thống, có và không có kiểm soát tiếp nhận (admission control).

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import heapq
from collections import deque

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]  # thứ tự màu cố định
N_YEU_CAU = 200_000  # số yêu cầu mỗi lần mô phỏng


def mot_may(lam, S, n, rng, tat_dinh=False):
    """Một máy, phục vụ theo thứ tự đến (FIFO). lam: yêu cầu mỗi giây, S: thời gian phục vụ trung bình (s).

    Trả về thời điểm đến và thời gian trong hệ thống (chờ + phục vụ) của từng yêu cầu, tính bằng giây.
    tat_dinh=True cho thời gian phục vụ cố định (M/D/1) thay vì phân phối mũ (M/M/1).
    """
    gian_cach = rng.exponential(1 / lam, n)          # đến theo Poisson: khoảng cách giữa hai lần đến là mũ
    phuc_vu = np.full(n, S) if tat_dinh else rng.exponential(S, n)
    den = np.cumsum(gian_cach)
    # Hồi quy Lindley: cho[i] = max(0, cho[i-1] + phuc_vu[i-1] - gian_cach[i]).
    # Viết qua tổng tích luỹ, NumPy tính cả mảng một lần mà không cần vòng lặp Python.
    buoc = np.concatenate(([0.0], phuc_vu[:-1] - gian_cach[1:]))
    tong = np.cumsum(buoc)
    cho = tong - np.minimum.accumulate(np.minimum(tong, 0.0))
    return den, cho + phuc_vu


def nhieu_may(den, S, c, rng, gioi_han=None):
    """c máy dùng chung một hàng đợi FIFO (M/M/c khi den là Poisson).

    gioi_han: số yêu cầu tối đa được phép chờ; yêu cầu đến khi hàng đã đầy bị từ chối ngay (503).
    Trả về thời gian trong hệ thống của từng yêu cầu, NaN cho yêu cầu bị từ chối.
    """
    phuc_vu = rng.exponential(S, len(den))
    ranh = [0.0] * c              # thời điểm mỗi máy rảnh, giữ thành một heap
    dang_cho = deque()            # thời điểm bắt đầu của những yêu cầu đã nhận mà chưa được phục vụ
    ket_qua = np.full(len(den), np.nan)
    for i, t in enumerate(den):
        while dang_cho and dang_cho[0] <= t:
            dang_cho.popleft()
        if gioi_han is not None and len(dang_cho) >= gioi_han:
            continue
        som_nhat = heapq.heappop(ranh)
        bat_dau = t if t > som_nhat else som_nhat
        heapq.heappush(ranh, bat_dau + phuc_vu[i])
        ket_qua[i] = bat_dau - t + phuc_vu[i]
        if bat_dau > t:
            dang_cho.append(bat_dau)
    return ket_qua


def den_poisson(lam, n, rng):
    return np.cumsum(rng.exponential(1 / lam, n))


print("đã nạp hai bộ mô phỏng: mot_may (M/M/1, M/D/1) và nhieu_may (M/M/c, có thể giới hạn hàng đợi)")

## 1. Định luật Little

Số yêu cầu trung bình nằm trong hệ thống bằng tốc độ đến (arrival rate) nhân với thời gian trung
bình mỗi yêu cầu nằm trong hệ thống:

$$Q_\text{req} = \lambda_\text{arr} \cdot T_\text{lat}.$$

Chương nói quan hệ này đúng với mọi hệ thống ổn định, bất kể phân phối của yêu cầu đến, phân phối
thời gian phục vụ hay chính sách lập lịch. Napkin math 1.2 lấy 1,000 QPS và 50 ms.

In [ ]:
lam_little = sach(1000, nguon="1,000 QPS · 50 ms")
T_little = sach(0.05, nguon="0.05 giây")
theo_sach("số yêu cầu trong hệ thống", lam_little * T_little, sach=50, nguon="0.05 giây · 50 yêu cầu")

# chiều ngược lại: 50 yêu cầu đồng thời, mỗi yêu cầu trung bình 10 ms
T_nguoc = sach(10, nguon="10 ms · 5,000 yêu cầu") / 1000
theo_sach("tốc độ đến suy ra (yêu cầu mỗi giây)", 50 / T_nguoc, sach=5000, nguon="5,000 yêu cầu")

Giờ kiểm định luật trên một hàng đợi mà ta không tính bằng công thức. Mô phỏng 200,000 yêu cầu đến
một máy ở 1,000 QPS, mỗi yêu cầu phục vụ trung bình 0.8 ms (giả định của sổ tay, mức sử dụng 80 %).
Rồi đếm số yêu cầu đang nằm trong hệ thống ở 20,000 thời điểm ngẫu nhiên: đó là những yêu cầu đã
đến mà chưa rời đi.

In [ ]:
rng = np.random.default_rng(42)
den, T = mot_may(lam_little, 0.0008, N_YEU_CAU, rng)
roi_di = np.sort(den + T)
thoi_diem = rng.uniform(den[1000], den[-1000], 20_000)  # bỏ hai đầu, nơi hàng đợi chưa ổn định
trong_he = np.searchsorted(den, thoi_diem, side="right") - np.searchsorted(roi_di, thoi_diem, side="right")
lam_do = len(den) / den[-1]
print(f"đếm trực tiếp: trung bình {trong_he.mean():.2f} yêu cầu trong hệ thống")
print(f"λ · T: {lam_do:,.0f} yêu cầu/s × {1000 * T.mean():.3f} ms = {lam_do * T.mean():.2f} yêu cầu")

Hai cách cho cùng một con số, sai khác chỉ do mẫu hữu hạn. Định luật Little không cần biết hàng
đợi trông ra sao bên trong. Đó vừa là sức mạnh vừa là giới hạn của nó: nó cho trung bình, nhưng
không cho biết độ trễ đổi thế nào khi tải tiến gần năng lực. Câu hỏi đó cần một mô hình của hàng
đợi.

## 2. Hàng đợi M/M/1: độ trễ bùng nổ gần điểm bão hoà

Với yêu cầu đến theo Poisson và thời gian phục vụ phân phối mũ, phương trình 5 và 6 của chương cho

$$T_\text{lat} = \frac{\text{thời gian phục vụ}}{1 - \rho_\text{serv}}, \qquad
T_\text{lat,p99} \approx \frac{\text{thời gian phục vụ}}{1 - \rho_\text{serv}} \cdot \ln\frac{1}{1 - 0.99}
\approx \frac{4.6 \cdot \text{thời gian phục vụ}}{1 - \rho_\text{serv}},$$

trong đó mức sử dụng $\rho_\text{serv}$ là tốc độ đến chia cho tốc độ phục vụ. Bảng 7 đọc phương
trình 5 ở năm mức sử dụng, với thời gian phục vụ 5 ms.

In [ ]:
S = sach(5, nguon="10 ms · 5 ms · 70 %") / 1000      # thời gian phục vụ, giây


def boi_trung_binh(rho):
    return 1 / (1 - rho)


theo_sach("bội số ở 50 %", boi_trung_binh(0.50), sach=2, nguon="50 % · 2 lần")
theo_sach("ví dụ ở 50 % (ms)", 1000 * S * boi_trung_binh(0.50), sach=10, nguon="2 lần · 10 ms")
theo_sach("bội số ở 70 %", boi_trung_binh(0.70), sach=3.3, nguon="70 % · 3.3 lần")
theo_sach("ví dụ ở 70 % (ms)", 1000 * S * boi_trung_binh(0.70), sach=17, nguon="3.3 lần · 17 ms")
theo_sach("bội số ở 80 %", boi_trung_binh(0.80), sach=5, nguon="80 % · 5 lần")
theo_sach("ví dụ ở 80 % (ms)", 1000 * S * boi_trung_binh(0.80), sach=25, nguon="5 lần · 25 ms")
theo_sach("bội số ở 90 %", boi_trung_binh(0.90), sach=10, nguon="90 % · 10 lần")
theo_sach("ví dụ ở 90 % (ms)", 1000 * S * boi_trung_binh(0.90), sach=50, nguon="10 lần · 50 ms")
theo_sach("bội số ở 95 %", boi_trung_binh(0.95), sach=20, nguon="95 % · 20 lần")
theo_sach("ví dụ ở 95 % (ms)", 1000 * S * boi_trung_binh(0.95), sach=100, nguon="20 lần · 100 ms")

LN100 = sach(4.6, nguon="4.6 · 4.6 lần")
theo_sach("ln(1 / (1 − 0.99))", math.log(1 / (1 - 0.99)), sach=4.6, nguon="4.6 · 4.6 lần")
theo_sach("p99 ở 70 %, bội số thời gian phục vụ", LN100 / (1 - 0.70), sach=15, nguon="70 % · 15 lần")
theo_sach("trung bình ở 70 % (ms)", 1000 * S / (1 - 0.70), sach=16.7, nguon="5 ms · 16.7 ms · 76.7 ms")
theo_sach("p99 ở 70 % (ms)", 1000 * LN100 * S / (1 - 0.70), sach=76.7, nguon="5 ms · 16.7 ms · 76.7 ms")

### Dự đoán

Phương trình 6 nói p99 của M/M/1 đúng bằng $\ln 100 \approx 4.6$ lần độ trễ trung bình, ở mọi mức
sử dụng. Đó là một xấp xỉ, hay là chính xác? Và nếu thời gian phục vụ không dao động chút nào, như
một mô hình đã biên dịch với hình dạng cố định (M/D/1), thì độ trễ trung bình ở 90 % giảm đi bao
nhiêu: một ít, một nửa, hay gần hết?

In [ ]:
MUC = [0.3, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]
dong = []
for rho in MUC:
    n = N_YEU_CAU if rho < 0.8 else 20 * N_YEU_CAU  # gần bão hoà hàng đợi "nhớ" lâu, cần mẫu dài hơn
    _, T_mm1 = mot_may(rho / S, S, n, np.random.default_rng(42))
    _, T_md1 = mot_may(rho / S, S, n, np.random.default_rng(42), tat_dinh=True)
    dong.append((rho, 1000 * T_mm1.mean(), 1000 * np.percentile(T_mm1, 99),
                 1000 * T_md1.mean(), 1000 * np.percentile(T_md1, 99)))
mm1 = np.array(dong)

print(" ρ    M/M/1 TB   pt.5     p99   ln100·TB   M/D/1 TB  P-K      p99")
for rho, tb, p99, tb_d, p99_d in mm1:
    pk = 1000 * (S + rho * S / (2 * (1 - rho)))  # Pollaczek–Khinchine cho phục vụ cố định
    print(f"{rho:.2f} {tb:9.1f} {1000 * S / (1 - rho):7.1f} {p99:7.1f} {math.log(100) * 1000 * S / (1 - rho):9.1f}"
          f" {tb_d:10.1f} {pk:5.1f} {p99_d:8.1f}")

Đọc bảng theo cột: trung bình mô phỏng của M/M/1 bám phương trình 5, và p99 mô phỏng bám
$\ln 100$ lần trung bình. Với M/M/1 theo thứ tự đến, thời gian trong hệ thống có phân phối mũ, nên
phân vị $q$ của nó đúng bằng $-\ln(1-q)$ lần trung bình: hệ số 4.6 của phương trình 6 là chính xác
cho mô hình này, phần "xấp xỉ" chỉ nằm ở việc làm tròn $\ln 100$. Sát bão hoà mô phỏng vẫn lệch vài phần
trăm dù đã dùng bốn triệu yêu cầu, vì hàng đợi gần bão hoà cần rất lâu mới quên trạng thái cũ.

Cột M/D/1 có trung bình theo công thức Pollaczek–Khinchine: phần **chờ** chỉ còn một nửa so với
M/M/1, nhưng vẫn tỉ lệ với $1/(1-\rho)$. Thời gian phục vụ ít biến động hơn làm đường cong thấp
xuống, không làm mất điểm khuỷu. Đây là lý do chương dùng M/M/1 như một biên thận trọng có lời
giải dạng đóng, rồi dặn phải đo phân phối thật.

In [ ]:
r = np.linspace(0.05, 0.955, 300)
fig, ax = plt.subplots(figsize=(8, 4.4))
ax.plot(r, 1000 * S / (1 - r), color=MAU[0], lw=2, label="M/M/1 trung bình, phương trình 5")
ax.plot(r, 1000 * LN100 * S / (1 - r), color=MAU[1], lw=2, label="M/M/1 p99, phương trình 6")
ax.plot(mm1[:, 0], mm1[:, 1], "o", color=MAU[0], ms=7, mec="white", mew=1.2, label="M/M/1 trung bình, mô phỏng")
ax.plot(mm1[:, 0], mm1[:, 2], "o", color=MAU[1], ms=7, mec="white", mew=1.2, label="M/M/1 p99, mô phỏng")
ax.plot(mm1[:, 0], mm1[:, 4], "s--", color=MAU[2], lw=1.2, ms=6, mec="white", mew=1.2,
        label="M/D/1 p99 mô phỏng (phục vụ cố định)")
ax.axvline(0.7, color="0.55", lw=0.8, ls=":")
ax.annotate("70 %: trung bình 3.3 lần,\np99 khoảng 15 lần thời gian phục vụ", (0.7, 1000 * LN100 * S / 0.3),
            xytext=(-150, 30), textcoords="offset points", fontsize=8,
            arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_ylim(0, 300)
ax.set_xlim(0, 1)
ax.set_xlabel("mức sử dụng ρ")
ax.set_ylabel("thời gian trong hệ thống (ms), phục vụ 5 ms")
ax.set_title("Độ trễ theo mức sử dụng: công thức và mô phỏng")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 3. Hai ngộ nhận và một cạm bẫy

**Ngộ nhận thứ nhất** cho rằng giảm độ trễ suy luận của mô hình sẽ giảm độ trễ người dùng thấy
theo cùng tỉ lệ. Kịch bản của chương: mức sử dụng 80 %, thời gian phục vụ 5 ms, rồi suy luận
xuống còn 2 ms trong khi lưu lượng giữ nguyên. Thời gian **chờ** của M/M/1 là
$\rho/(1-\rho)$ lần thời gian phục vụ.

**Dự đoán:** mô hình nhanh hơn 2.5 lần. Người dùng thấy nhanh hơn bao nhiêu lần?

In [ ]:
rho_cu = sach(80, nguon="80 % · 5 ms · 20 ms") / 100
S_cu = sach(5, nguon="80 % · 5 ms · 20 ms")
S_moi = sach(2, nguon="20 ms · 2 ms · 80")
rho_moi = rho_cu * S_moi / S_cu        # cùng tốc độ đến, phục vụ nhanh hơn


def cho_mm1(rho, s):
    return rho / (1 - rho) * s


theo_sach("chờ trung bình lúc đầu (ms)", cho_mm1(rho_cu, S_cu), sach=20, nguon="80 % · 5 ms · 20 ms")
theo_sach("mức sử dụng mới (%)", 100 * rho_moi, sach=32, nguon="80 · 32 %")
theo_sach("chờ trung bình mới (ms)", cho_mm1(rho_moi, S_moi), sach=0.9, nguon="0.9 ms · 21.2 lần")
giam_cho = cho_mm1(rho_cu, S_cu) / cho_mm1(rho_moi, S_moi)
print(f"chờ giảm {giam_cho:.4f} lần; sách ghi {sach(21.2, nguon='0.9 ms · 21.2 lần')} lần (xem ghi chú dưới)")
T_cu, T_moi = S_cu / (1 - rho_cu), S_moi / (1 - rho_moi)
theo_sach("độ trễ trong hệ thống lúc đầu (ms)", T_cu, sach=25, nguon="21.2 lần · 25 · 2.9 ms")
theo_sach("độ trễ trong hệ thống mới (ms)", T_moi, sach=2.9, nguon="25 · 2.9 ms")
theo_sach("tăng tốc ở mức hệ thống", T_cu / T_moi, sach=8.5, nguon="2.9 ms · 8.5 lần")
theo_sach("tăng tốc ở mức mô hình", S_cu / S_moi, sach=2.5, nguon="8.5 lần · 2.5 lần")

Thời gian chờ giảm đúng 21.25 lần: 20 ms chia cho $0.32/0.68 \times 2 \approx 0.941$ ms. Sách ghi 21.2 lần, tức
làm tròn 21.25 xuống còn một chữ số thập phân. Hai con số chỉ khác nhau ở cách làm tròn, nên ô trên in cả
hai thay vì đối chiếu.

Mô hình nhanh hơn làm hai việc cùng lúc: nó cắt thời gian phục vụ, và nó kéo mức sử dụng xuống
khỏi điểm khuỷu, nơi thời gian chờ chiếm phần lớn. Phần thứ hai mới là phần lớn.

**Cạm bẫy thứ nhất** là chạy hạ tầng ở mức sử dụng cao để tối đa hiệu quả chi phí. Với cùng lưu
lượng, số máy cần có tỉ lệ nghịch với mức sử dụng mục tiêu.

In [ ]:
rho_70 = sach(70, nguon="10 lần · 70") / 100
rho_90 = sach(90, nguon="2.5 lần · 90 %") / 100
theo_sach("ở 90 %: bội số thời gian phục vụ", boi_trung_binh(rho_90), sach=10, nguon="90 % · 10 lần")
theo_sach("70 % → 90 %: chi phí hạ tầng giảm (%)", 100 * (1 - rho_70 / rho_90), sach=22.2, nguon="70 · 22.2 %")
gap = boi_trung_binh(rho_90) / boi_trung_binh(rho_70)
assert abs(gap - 3) < 1e-9   # "tăng gấp ba": (1/0.1) ÷ (1/0.3)
print(f"độ trễ trung bình tăng {gap:.1f} lần")
theo_sach("p99 ở 70 % (ms)", LN100 * S_cu / (1 - rho_70), sach=76.7, nguon="22.2 % · 76.7")
theo_sach("p99 ở 90 % (ms)", LN100 * S_cu / (1 - rho_90), sach=230, nguon="76.7 · 230 ms")

Bớt 22.2 % máy, đổi lại độ trễ trung bình gấp ba và p99 từ khoảng 76.7 ms lên khoảng 230 ms. Hai
điều cần mang theo ở phần tổng kết của chương cũng chính là bảng 7: 5 lần thời gian phục vụ ở 80 %,
10 lần ở 90 %.

**Cạm bẫy thứ hai** là đánh giá hệ thống bằng độ trễ trung bình. Ở 70 % với 5 ms, trung bình là
16.7 ms còn p99 là 76.7 ms. Bảng mô phỏng ở phần 2 cho thấy khoảng cách ấy có thật, không phải sản
phẩm của công thức.

In [ ]:
theo_sach("80 %: bội số", boi_trung_binh(sach(80, nguon="5 lần · 80 %") / 100), sach=5, nguon="5 lần · 80 %")
theo_sach("90 %: bội số", boi_trung_binh(sach(90, nguon="10 lần · 90 %") / 100), sach=10, nguon="10 lần · 90 %")
theo_sach("70 %: p99 ÷ trung bình", (LN100 * S_cu / (1 - rho_70)) / (S_cu / (1 - rho_70)), sach=4.6, nguon="76.7 ms · 4.6 lần")
o70 = int(np.argmin(np.abs(mm1[:, 0] - 0.7)))
print(f"mô phỏng ở 70 %: trung bình {mm1[o70, 1]:.1f} ms, p99 {mm1[o70, 2]:.1f} ms")

## 4. Hoạch định năng lực cho ResNet-50

Napkin math 1.3 thiết kế một hệ thống với đích p99 là 50 ms, lưu lượng đỉnh 5,000 QPS, và thời gian
phục vụ 5 ms (TensorRT FP16). Năm bước của sách:

In [ ]:
SLO = sach(50, nguon="50 ms · 5,000 QPS")
LUU_LUONG = sach(5000, nguon="5,000 QPS · 5 ms")
S_gpu = sach(5, nguon="5,000 QPS · 5 ms")
HE_SO = sach(4.6, nguon="5 ms · 4.6 · 0.54")
MOT_GPU = sach(200, nguon="200 yêu cầu")          # yêu cầu mỗi giây của một GPU, 1 / 5 ms
assert MOT_GPU == 1000 / S_gpu

rho_an_toan = 1 - HE_SO * S_gpu / SLO                                   # bước 1
theo_sach("mức sử dụng an toàn", rho_an_toan, sach=0.54, nguon="4.6 · 0.54 · 54 %")
mu_can = LUU_LUONG / rho_an_toan                                         # bước 2
theo_sach("tốc độ phục vụ cần có (yêu cầu/s)", mu_can, sach=9259.3, nguon="9,259.3 yêu cầu")
gpu_le = mu_can / MOT_GPU                                                # bước 3
theo_sach("số GPU, chưa làm tròn", gpu_le, sach=46.3, nguon="200 yêu cầu · 46.3")
gpu_47 = math.ceil(gpu_le)
theo_sach("số GPU, làm tròn lên", gpu_47, sach=47, nguon="46.3 · 47 GPU")
du_dia = sach(30, nguon="47 GPU · 30 %") / 100                           # bước 4
theo_sach("47 × 1.3", gpu_47 * (1 + du_dia), sach=61.1, nguon="47 · 61.1 · 62")
gpu_62 = math.ceil(gpu_47 * (1 + du_dia))
theo_sach("số GPU sau dư địa", gpu_62, sach=62, nguon="61.1 · 62 · 61")
sau_su_co = LUU_LUONG / (MOT_GPU * (gpu_62 - 1))                         # bước 5, N+1
theo_sach("mức sử dụng khi mất một GPU (%)", 100 * sau_su_co, sach=41, nguon="200 · 41 %")
gpu_n2 = math.ceil(gpu_le) + 2                                           # N+2, cùng ngưỡng
theo_sach("N+2 với cùng ngưỡng", gpu_n2, sach=49, nguon="49 GPU")
theo_sach("N+2 vẫn giữ 30 % dư địa", gpu_62 + 2, sach=64, nguon="64 GPU")

Sách nói rõ phép tính này coi mỗi GPU là một hàng đợi M/M/1 riêng. Điều đó đúng khi bộ cân bằng
tải (load balancer) gửi mỗi yêu cầu tới một GPU chọn ngẫu nhiên: tách một dòng Poisson một cách
ngẫu nhiên cho ra những dòng Poisson nhỏ hơn, độc lập với nhau. Khi đó với 47 GPU, mỗi GPU nhận
$5{,}000/47$ yêu cầu mỗi giây và p99 phải nằm ngay dưới 50 ms.

Chương cũng nói mức cải thiện p99 thật tuỳ vào chính sách phân phối. Một cách khác là **một hàng
đợi chung**: yêu cầu xếp một hàng, GPU nào rảnh thì lấy yêu cầu kế tiếp (M/M/c).

**Dự đoán:** với hàng đợi chung, cần bao nhiêu GPU để p99 dưới 50 ms? Vẫn khoảng 47, hay ít hơn hẳn?
Số tối thiểu tuyệt đối là 25, vì 25 × 200 = 5,000.

In [ ]:
cac_gpu = [26, 27, 28, 30, 35, 40, 47, 55, 62]
p99_ngau_nhien, p99_chung = [], []
for g in cac_gpu:
    # định tuyến ngẫu nhiên: mỗi GPU là một M/M/1 với λ/g, mô phỏng một GPU là đủ
    _, T_g = mot_may(LUU_LUONG / g, S_gpu / 1000, N_YEU_CAU, np.random.default_rng(42))
    p99_ngau_nhien.append(1000 * np.percentile(T_g, 99))
    T_c = nhieu_may(den_poisson(LUU_LUONG, N_YEU_CAU, np.random.default_rng(42)), S_gpu / 1000, g,
                    np.random.default_rng(7))
    p99_chung.append(1000 * np.percentile(T_c, 99))
    print(f"{g:>2} GPU, ρ = {LUU_LUONG / (g * MOT_GPU):.2f}: p99 ngẫu nhiên {p99_ngau_nhien[-1]:6.1f} ms"
          f" (công thức {HE_SO * S_gpu / (1 - LUU_LUONG / (g * MOT_GPU)):6.1f}),  hàng đợi chung {p99_chung[-1]:5.1f} ms")
it_nhat = next(g for g, p in zip(cac_gpu, p99_chung) if p <= SLO)
print(f"hàng đợi chung: {it_nhat} GPU đã đạt p99 ≤ {SLO} ms trong mô phỏng này")

Kết quả cho thấy hai điều. Với định tuyến ngẫu nhiên, 47 GPU cho p99 đúng sát 50 ms như bước 3
của sách tính. Với hàng đợi chung, số GPU giảm hẳn, vì một GPU rảnh không bao giờ đứng nhìn một
GPU khác có hàng chờ. Đó là lý do bộ cân bằng tải thật hay gửi yêu cầu tới bản sao (replica) có ít
việc nhất thay vì chọn bừa.

Nhưng một hệ thống chạy ở 96 % mức sử dụng không có dư địa. Câu hỏi đúng không phải "bao nhiêu GPU
đạt SLO hôm nay" mà "lưu lượng tăng bao nhiêu thì SLO vỡ". Ô dưới đẩy lưu lượng lên tới 1.4 lần
cho ba đội máy.

In [ ]:
he_so_tang = np.round(np.arange(1.0, 1.41, 0.05), 2)
doi_may = [("26 GPU, hàng đợi chung (mô phỏng)", 26, "chung"), ("47 GPU, ngẫu nhiên (phương trình 6)", 47, "ngẫu nhiên"),
           ("62 GPU, ngẫu nhiên, kế hoạch của sách", 62, "ngẫu nhiên")]
duong = {}
for ten, g, kieu in doi_may:
    ys = []
    for k in he_so_tang:
        lam = LUU_LUONG * k
        if lam >= g * MOT_GPU:
            ys.append(np.nan)            # không ổn định: hàng đợi dài ra mãi
            continue
        if kieu == "chung":
            T_k = nhieu_may(den_poisson(lam, N_YEU_CAU, np.random.default_rng(42)), S_gpu / 1000, g,
                            np.random.default_rng(7))
            ys.append(1000 * np.percentile(T_k, 99))
        else:
            ys.append(HE_SO * S_gpu / (1 - lam / (g * MOT_GPU)))   # M/M/1 mỗi GPU, công thức chính xác
    duong[ten] = np.array(ys)
    vo = next((k for k, y in zip(he_so_tang, ys) if not (y <= SLO)), None)
    print(f"{ten}: SLO vỡ ở {'lưu lượng × ' + str(vo) if vo else 'quá × 1.4'}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2))
ax1.plot(cac_gpu, p99_ngau_nhien, "o-", color=MAU[0], lw=2, ms=6, mec="white", label="định tuyến ngẫu nhiên (M/M/1 mỗi GPU)")
ax1.plot(cac_gpu, p99_chung, "s-", color=MAU[1], lw=2, ms=6, mec="white", label="một hàng đợi chung (M/M/c)")
ax1.axhline(SLO, color="0.4", lw=1, ls="--")
ax1.text(26, SLO - 6, "SLO p99 50 ms", fontsize=8, color="0.3")
for g, nhan, lech in ((47, "47 GPU: bước 3", (6, 14)), (62, "62 GPU: sau dư địa", (-6, -14))):
    ax1.annotate(nhan, (g, p99_ngau_nhien[cac_gpu.index(g)]), xytext=lech, textcoords="offset points",
                 fontsize=8, ha="left" if lech[0] > 0 else "right",
                 arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax1.set_ylim(0, 120)
ax1.set_xlabel("số GPU, 5,000 QPS")
ax1.set_ylabel("p99 (ms, mô phỏng)")
ax1.set_title("Cùng lưu lượng, hai cách phân phối yêu cầu")
ax1.legend(loc="upper right", fontsize=8)
for (ten, ys), mau in zip(duong.items(), (MAU[1], MAU[0], MAU[2])):
    ax2.plot(he_so_tang, ys, "o-", color=mau, lw=2, ms=5, mec="white", label=ten)
ax2.axhline(SLO, color="0.4", lw=1, ls="--")
ax2.axvline(26 * MOT_GPU / LUU_LUONG, color=MAU[1], lw=1, ls=":")
ax2.annotate("26 GPU bão hoà ở × 1.04:\nhàng đợi dài ra mãi", (26 * MOT_GPU / LUU_LUONG, 30), xytext=(10, -40),
             textcoords="offset points", fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax2.set_ylim(0, 120)
ax2.set_xlabel("lưu lượng so với 5,000 QPS (lần)")
ax2.set_ylabel("p99 (ms)")
ax2.set_title("Dư địa: lưu lượng tăng bao nhiêu thì SLO vỡ")
ax2.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Đội 26 GPU dùng hàng đợi chung đạt SLO đúng ở 5,000 QPS rồi sụp gần như ngay khi lưu lượng nhích
lên, vì nó đã ở sát bão hoà; đường của nó dừng ở chỗ hàng đợi không còn ổn định. Đội 47 GPU của bước
3 vỡ SLO ngay khi lưu lượng vượt đỉnh dự kiến. Đội 62 GPU của sách giữ được SLO tới khi lưu lượng tăng
chừng ba mươi phần trăm, đúng phần dư địa đã mua. Mô hình nào cũng cho thấy cùng một điều: SLO được
giữ bằng dư địa, không phải chỉ bằng suy luận nhanh.

## 5. Toả ra nhiều máy: đuôi của một máy thành phần giữa của cả hệ thống

Dean và Barroso chỉ ra rằng khi một yêu cầu toả ra nhiều máy chủ và phải chờ tất cả trả lời, xác
suất gặp ít nhất một phản hồi chậm tăng rất nhanh theo số máy. Sách không đưa con số nào cho hiện
tượng này, nên mọi con số dưới đây là phép tính của sổ tay. Theo định nghĩa, mỗi máy có 1 % phản hồi
chậm hơn p99 của nó; với $N$ máy độc lập,

$$\Pr(\text{ít nhất một phản hồi chậm hơn p99}) = 1 - 0.99^N.$$

**Dự đoán:** với 100 máy, xác suất đó là khoảng 1 %, khoảng 10 %, hay hơn một nửa?

In [ ]:
p_cham = 1 - 0.99            # định nghĩa của p99, không phải một số liệu
for N in (1, 10, 100, 1000):
    print(f"{N:>5} máy: {1 - (1 - p_cham) ** N:6.1%} yêu cầu gặp ít nhất một phản hồi chậm hơn p99 của một máy")

# kiểm bằng mô phỏng: lấy phân phối độ trễ thật của một máy M/M/1 ở 70 %
_, T70 = mot_may(0.7 / S, S, N_YEU_CAU, np.random.default_rng(42))
nguong = np.percentile(T70, 99)
rng = np.random.default_rng(3)
cham_nhat = rng.choice(T70, size=(20_000, 100)).max(axis=1)   # 20,000 yêu cầu, mỗi yêu cầu chờ 100 máy
print(f"mô phỏng, 100 máy: {np.mean(cham_nhat > nguong):.1%}; trung vị của yêu cầu toả ra là"
      f" {1000 * np.median(cham_nhat):.0f} ms, so với p99 của một máy {1000 * nguong:.0f} ms")

N_ = np.unique(np.logspace(0, 3.3, 200).astype(int))
fig, ax = plt.subplots(figsize=(7.5, 4))
for q, mau, nhan in ((0.99, MAU[0], "chậm hơn p99 của một máy"), (0.999, MAU[1], "chậm hơn p99.9 của một máy")):
    ax.semilogx(N_, 100 * (1 - q ** N_), color=mau, lw=2, label=nhan)
for N in (10, 100):
    y = 100 * (1 - 0.99 ** N)
    ax.plot(N, y, "o", color=MAU[0], ms=7, mec="white", mew=1.2)
    ax.annotate(f"{N} máy: {y:.1f} %", (N, y), xytext=(-75, 10), textcoords="offset points", fontsize=8)
ax.set_ylim(0, 105)
ax.set_xlabel("số máy một yêu cầu toả ra (thang log)")
ax.set_ylabel("yêu cầu gặp ít nhất một phản hồi chậm (%)")
ax.set_title("Toả ra nhiều máy: phép tính của sổ tay, sách không nêu số")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Với 100 máy, gần hai phần ba yêu cầu gặp ít nhất một phản hồi chậm hơn p99 của một máy, và trung
vị của yêu cầu toả ra đã nằm quanh p99 của một máy. Phần đuôi hiếm của từng máy thành trải nghiệm
thường ngày của người dùng. Đây là lý do chương nói đuôi của từng máy quan trọng ngay cả khi hệ
thống mới chạy trên một máy: nó sẽ cộng dồn khi hệ thống mở rộng.

Một trong các kỹ thuật chịu được đuôi là gửi yêu cầu dự phòng (request hedging): khi một yêu cầu
chưa xong sau ngưỡng phân vị 95 lịch sử, gửi thêm một bản tới máy khác và dùng phản hồi đến trước.
Với phân phối ổn định, ngưỡng ấy chạm tới đúng phần yêu cầu nằm trên p95:

In [ ]:
p95 = sach(95, nguon="95 · 5 %")
nguong95 = np.percentile(T70, p95)
theo_sach("phần yêu cầu được gửi dự phòng (%)", 100 * np.mean(T70 > nguong95), sach=5, nguon="95 · 5 %")

## 6. Một đợt tăng vọt, và kiểm soát tiếp nhận

Ví dụ 1.1 của sách là kịch bản Black Friday: lưu lượng tăng 10 lần lên 10,000 QPS, mức sử dụng tiến
tới 100 %, hàng đợi dài ra phi tuyến và độ trễ lên tới 10 giây. Phần này dựng một phiên bản nhỏ của
kịch bản ấy, phỏng theo phần B và E của phòng thí nghiệm đi kèm chương. Mọi thông số là **giả định
của sổ tay**: bốn worker, mỗi worker phục vụ trung bình 5 ms (năng lực 800 yêu cầu mỗi giây), lưu
lượng nền 500 yêu cầu mỗi giây, rồi một đợt 1,000 yêu cầu mỗi giây kéo dài 5 giây.

Trong đợt tăng, tốc độ đến vượt năng lực, nên không còn trạng thái ổn định và định luật Little không
áp dụng được: phần tồn đọng cứ tích lại. Chương đưa một điểm xuất phát cho kiểm soát tiếp nhận: hàng
đợi chứa hai tới ba lượng phục vụ cho mỗi worker, tức 8 tới 12 yêu cầu đang chờ với bốn worker. Yêu
cầu đến khi hàng đã đầy bị từ chối ngay bằng phản hồi 503.

**Dự đoán:** không giới hạn hàng đợi, bao nhiêu phần trăm yêu cầu còn kịp hạn 50 ms? Giới hạn 12 yêu
cầu thì từ chối bao nhiêu phần trăm, và bao nhiêu phần trăm kịp hạn?

In [ ]:
SLO_BAO = SLO / 1000                            # dùng lại đích 50 ms của napkin math 1.3
CO_NEN, CO_DOT = 500, 1000                      # yêu cầu mỗi giây, giả định
WORKER, S_W = 4, 0.005
truoc, dot, sau = 20.0, 5.0, 20.0               # giây
rng = np.random.default_rng(42)
den_bao = np.concatenate([np.sort(rng.uniform(a, a + d, rng.poisson(lam * d)))
                          for lam, a, d in ((CO_NEN, 0, truoc), (CO_DOT, truoc, dot), (CO_NEN, truoc + dot, sau))])

ton_dong = (CO_DOT - WORKER / S_W) * dot
print(f"năng lực {WORKER / S_W:.0f} yêu cầu/s; trong đợt tăng tồn đọng thêm khoảng {ton_dong:,.0f} yêu cầu,"
      f" tức chờ thêm khoảng {ton_dong / (WORKER / S_W):.2f} s ở cuối đợt")

gh_thap = sach(8, trich="with four workers, that corresponds to 8 to 12 queued requests")
gh_cao = sach(12, nguon="503 · 12 yêu cầu")
chinh_sach = {"không giới hạn": None, f"giới hạn {gh_thap}": gh_thap, f"giới hạn {gh_cao}": gh_cao}
ket_qua_bao = {}
for ten, gh in chinh_sach.items():
    lat = nhieu_may(den_bao, S_W, WORKER, np.random.default_rng(7), gioi_han=gh)
    ket_qua_bao[ten] = lat
    nhan = ~np.isnan(lat)
    kip = nhan & (lat <= SLO_BAO)
    print(f"{ten:>15}: từ chối {1 - nhan.mean():5.1%} · p99 của yêu cầu được nhận {1000 * np.nanpercentile(lat, 99):7.1f} ms"
          f" · lâu nhất {1000 * np.nanmax(lat):7.1f} ms · kịp hạn 50 ms {kip.mean():6.1%} tổng số yêu cầu")

In [ ]:
CUA = 0.5                                        # giây mỗi ô thời gian
bien = np.arange(0, truoc + dot + sau + CUA, CUA)
o = np.digitize(den_bao, bien) - 1
giua = bien[:-1] + CUA / 2
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5.6), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
for ax in (ax1, ax2):
    ax.axvspan(truoc, truoc + dot, color="0.92", lw=0)
for (ten, lat), mau in zip(ket_qua_bao.items(), (MAU[1], MAU[2], MAU[0])):
    p99_o = [1000 * np.nanpercentile(lat[o == k], 99) if np.any(~np.isnan(lat[o == k])) else np.nan
             for k in range(len(giua))]
    tu_choi = [100 * np.mean(np.isnan(lat[o == k])) for k in range(len(giua))]
    ax1.plot(giua, p99_o, color=mau, lw=2, label=ten)
    if ten != "không giới hạn":
        ax2.plot(giua, tu_choi, color=mau, lw=2, label=ten)
ax1.axhline(1000 * SLO_BAO, color="0.3", lw=1, ls="--")
ax1.text(0.5, 1000 * SLO_BAO * 1.2, "hạn 50 ms", fontsize=8, color="0.3")
ax1.text(truoc + dot / 2, 2, "đợt tăng\n1,000 yêu cầu/s", ha="center", va="bottom", fontsize=8, color="0.3")
ax1.set_yscale("log")
ax1.set_ylim(1, 5e3)
ax1.set_ylabel("p99 mỗi 0.5 s của yêu cầu\nđược nhận (ms, thang log)")
ax1.set_title("Một đợt tăng vọt qua bốn worker (giả định của sổ tay, mô phỏng)")
ax1.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
ax2.set_ylabel("bị từ chối (%)")
ax2.set_xlabel("thời điểm đến (s)")
ax2.set_xlim(0, truoc + dot + sau)
ax2.set_ylim(0, 40)
plt.tight_layout()
plt.show()

Không giới hạn hàng đợi, hệ thống nhận hết mọi yêu cầu và trả lời tất cả, nhưng phần tồn đọng tích
trong đợt tăng kéo độ trễ lên cỡ giây, và còn kéo dài vài giây **sau** khi đợt tăng đã qua, trong lúc
hàng đợi xả dần. Mọi yêu cầu trong khoảng đó trễ hạn, dù chúng đến lúc lưu lượng đã bình thường. Với
giới hạn hàng đợi, một phần nhỏ yêu cầu bị từ chối ngay, còn những yêu cầu được nhận giữ độ trễ quanh
mức bình thường. Đổi một ít thông lượng lấy độ trễ của những yêu cầu được nhận là đúng điều chương
mô tả, và tính trên tổng số yêu cầu, số kịp hạn còn nhiều hơn so với khi không giới hạn.

Sách nói ngưỡng 8 tới 12 là một giả thuyết để kiểm, không phải một ngưỡng dùng được ở mọi nơi. Phần
Thử thêm dưới đây cho bạn kiểm nó.

## Thử thêm

1. Ở ô `burst`, đổi `CO_DOT` thành `850`, chỉ vượt năng lực một chút. Hệ thống không giới hạn còn
   trễ hạn nhiều không? Rồi thử `gioi_han` bằng 2 và bằng 50 trong `chinh_sach`: ngưỡng quá chặt và
   quá lỏng hỏng theo hai cách nào?
2. Ở ô `lib`, đổi dòng `phuc_vu = rng.exponential(S, len(den))` của `nhieu_may` thành
   `phuc_vu = np.full(len(den), S)` (phục vụ cố định, M/D/c), rồi chạy lại ô `routing`. Với phục vụ
   không dao động, hàng đợi chung cần bao nhiêu GPU, và đội 26 GPU còn vỡ SLO sớm như trước không?
3. Ở ô `fanout`, thay ngưỡng p99 bằng p99.9 của một máy (`np.percentile(T70, 99.9)`) và tính lại phần
   mô phỏng. Muốn chưa tới 10 % yêu cầu toả ra 100 máy gặp phản hồi chậm, mỗi máy phải giữ phân vị
   nào?

## Tóm lại

* Định luật Little ($Q = \lambda T$) đúng với mọi hệ thống ổn định, và mô phỏng cho cùng con số với
  phép đếm trực tiếp; nó cho trung bình, không cho phần đuôi.
* Với M/M/1, độ trễ trung bình là $S/(1-\rho)$ và p99 đúng bằng $\ln 100 \approx 4.6$ lần trung bình:
  5 lần thời gian phục vụ ở 80 %, 10 lần ở 90 %. Phục vụ cố định (M/D/1) hạ đường cong nhưng không
  xoá điểm khuỷu.
* Mô hình nhanh hơn kéo mức sử dụng xuống khỏi điểm khuỷu, nên độ trễ người dùng thấy có thể giảm
  nhiều hơn tỉ lệ tăng tốc của mô hình; chạy ở 90 % thì ngược lại.
* Napkin math 1.3 cho 62 GPU khi mỗi GPU là một M/M/1; một hàng đợi chung cần ít GPU hơn hẳn, nhưng
  chỉ dư địa mới giữ được SLO khi lưu lượng tăng.
* Khi tốc độ đến vượt năng lực, phần tồn đọng tích lại và kéo trễ hạn cả sau đợt tăng; kiểm soát
  tiếp nhận đổi một ít yêu cầu bị từ chối lấy độ trễ của những yêu cầu được nhận.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Phục vụ mô hình — mỗi yêu cầu một ngân sách độ trễ](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch13-model-serving/), dựng từ chương
[*Model Serving*](https://mlsysbook.ai/vol1/model_serving/model_serving.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_13_model_serving.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.